# 02 — Shazam Pipeline

Baut den Shazam-Fingerprint-Index auf `dry_ref` (50 Songs) auf und läuft
alle **2 046 Queries** aus dem Manifest durch (In-DB + OOD zusammen).

**Shazam-Konfiguration (Wang 2003):**
- Sample Rate: 22 050 Hz (interne Resampling durch shazam_fingerprint)
- Kombinatorisches Hashing: Fan-Out 10, 32-Bit Hash
- Match-Threshold: 10 Histogram-Peaks

**Abhängigkeiten:** NB 00 (`data/partitions/`), NB 01 (`data/queries/manifest_dry.csv`)
**Ausgabe:** `results/dry_run/shazam_raw.csv`, `results/dry_run/shazam_efficiency.json`


In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1. Imports und Pfade

`shazam_fingerprint/` muss auf `sys.path` liegen damit die Shazam-Imports
funktionieren. `shazam_pipeline.py` enthält `build_shazam_index()` und
`run_shazam_query()`.


In [ ]:
import os, sys, json, time
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from tqdm import tqdm
from shazam_pipeline import build_shazam_index, run_shazam_query
from metrics import classify_result
from utils import load_fma_metadata, load_partitions
from run_config import cfg

RUN_MODE = cfg.run_mode

# ── Mode-dependent paths ──────────────────────────────────────────────────────
MANIFEST_PATH  = PROJECT_ROOT / "data" / "queries" / cfg.manifest_name
RESULTS_DIR    = PROJECT_ROOT / "results" / cfg.results_subdir
REF_KEY        = cfg.ref_key   # "dry_ref" or "live_ref"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE      = {RUN_MODE!r}")
print(f"REF_KEY       = {REF_KEY!r}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")

## 2. Voraussetzungen prüfen

Stellt sicher, dass NB 00 und NB 01 vollständig ausgeführt wurden, bevor
der zeitintensive Index-Build beginnt.


In [ ]:
partitions_dir = PROJECT_ROOT / "data" / "partitions"

assert MANIFEST_PATH.exists(), f"Manifest fehlt: {MANIFEST_PATH} — NB 01 ausführen!"
assert (partitions_dir / f"{REF_KEY}.json").exists(),     f"{REF_KEY}.json fehlt — NB 00 ausführen!"

manifest   = pd.read_csv(MANIFEST_PATH)
partitions = load_partitions(partitions_dir)
ref_ids    = partitions[REF_KEY]

print(f"Manifest : {len(manifest)} Zeilen")
print(f"{REF_KEY}: {len(ref_ids)} Tracks")

## 3. FMA-Metadaten laden

Die Filepath-Spalte aus `track_df` wird benötigt um den Shazam-Index
auf den Referenz-MP3-Dateien aufzubauen (nicht auf den Query-WAVs).


In [ ]:
track_df = load_fma_metadata(PROJECT_ROOT / "data" / "fma_medium")
missing = [tid for tid in ref_ids if tid not in track_df.index]
assert len(missing) == 0, f"Tracks nicht in Metadaten: {missing}"
print(f"Metadaten geladen: {len(track_df)} Tracks verfügbar")
print(f"Baue Index auf {len(ref_ids)} Ref-Tracks ({REF_KEY}) ...")

## 4. Shazam-Index aufbauen (dry_ref)

`build_shazam_index()` verarbeitet die Referenz-MP3-Dateien der `dry_ref`-Tracks:
`load_audio → compute_spectrogram → find_peaks → generate_fingerprints → db.insert`

`song_id` = nullgepadded 6-stelliger String (z. B. `"000052"` für track_id=52).


In [ ]:
print("Baue Shazam-Index auf ...")
t_build_start = time.perf_counter()
shazam_db, build_stats = build_shazam_index(ref_ids, track_df)
build_time_s = time.perf_counter() - t_build_start

db_stats = shazam_db.get_stats()
index_size_mb = db_stats["memory_mb"]

print(f"Build-Zeit:   {build_time_s:.1f} s")
print(f"Index-Größe:  {index_size_mb:.1f} MB")
print(f"Hashes total: {build_stats['total_hashes']:,}")
print(f"Fehlgeschlagen: {build_stats['failed']}")

In [ ]:
print(db_stats)
shazam_db.save("database")

## 5. Alle Queries ausführen (In-DB + OOD)

Für jede Zeile im Manifest: Query stellen, Zeit messen, Ergebnis klassifizieren.

- `pred_id = None` → kein Match gefunden (niemals 0 oder -1)
- `classify_result(pred_id, ref_track_id, is_ood)` → "TP"/"FP"/"FN"/"TN"
- `query_time_ms` = Gesamtzeit inkl. Fingerprinting (via `time.perf_counter()`)


In [ ]:
raw_rows = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc="Shazam"):
    pred_id, score, q_ms = run_shazam_query(row.query_path, shazam_db)

    ref_id = None if pd.isna(row.ref_track_id) else int(row.ref_track_id)
    result_class = classify_result(pred_id, ref_id, bool(row.is_ood))

    raw_rows.append({
        "system":        "shazam",
        "track_id":      int(row.track_id),
        "ref_track_id":  ref_id,
        "is_ood":        bool(row.is_ood),
        "predicted_id":  pred_id,
        "score":         score,
        "result_class":  result_class,
        "query_time_ms": q_ms,
        "group":         row.group,
        "condition":     row.condition,
        "duration_sec":  float(row.duration_sec),
    })
print(f"Queries abgeschlossen: {len(raw_rows)}")


## 6. Raw-Ergebnisse speichern

`predicted_id` ist nullable Int64 damit `None`-Werte korrekt als leere
Felder in der CSV landen.


In [ ]:
results_df = pd.DataFrame(raw_rows)
results_df["predicted_id"] = results_df["predicted_id"].astype(pd.Int64Dtype())
results_df["ref_track_id"] = results_df["ref_track_id"].astype(pd.Int64Dtype())

out_csv = RESULTS_DIR / "shazam_raw.csv"
results_df.to_csv(out_csv, index=False)
print(f"Gespeichert: {out_csv}  ({len(results_df)} Zeilen)")
print(results_df.head(3).to_string())

## 7. Effizienz-Metriken speichern

`build_time_s` und `index_size_mb` werden separat gespeichert für die
Effizienz-Tabelle in NB 06. Timing ist mit `time.perf_counter()` gemessen.


In [ ]:
efficiency = {
    "system":          "shazam",
    "run_mode":        RUN_MODE,
    "build_time_s":    round(build_time_s, 3),
    "index_size_mb":   round(index_size_mb, 3),
    "n_ref_tracks":    build_stats["processed"],
    "total_hashes":    build_stats["total_hashes"],
    "mean_query_ms":   round(results_df["query_time_ms"].mean(), 3),
    "median_query_ms": round(results_df["query_time_ms"].median(), 3),
}
eff_path = RESULTS_DIR / "shazam_efficiency.json"
with open(eff_path, "w") as fh:
    json.dump(efficiency, fh, indent=2)
print(f"Effizienz gespeichert: {eff_path}")
print(json.dumps(efficiency, indent=2))

## 8. Sanity Checks

Verifikation der Ergebnis-Qualität bevor NB 06 die Daten verwendet.


In [ ]:
n_total = len(results_df)
classes = results_df["result_class"].value_counts()
print(f"Gesamt: {n_total} Queries")
print("Ergebnisklassen:")
print(classes.to_string())

# Alle vier Klassen müssen vorhanden sein
for cls in ["TP", "FN", "FP", "TN"]:
    if cls not in classes:
        print(f"  WARNUNG: Klasse {cls} fehlt komplett!")
    else:
        print(f"  {cls}: {classes[cls]} ✓")


In [ ]:
# OOD-Einträge dürfen keine ref_track_id haben
ood = results_df[results_df["is_ood"]]
assert ood["ref_track_id"].isna().all(), "OOD-Einträge haben ref_track_id!"

# pred_id=None ist erlaubt, 0 oder -1 wäre ein Bug
invalid_pred = results_df["predicted_id"].dropna()
assert (invalid_pred > 0).all(), "predicted_id enthält 0 oder negative Werte!"

# Ergebnisklassen-Set
assert set(results_df["result_class"]).issubset({"TP","FP","FN","TN"}),     "Unbekannte Ergebnisklassen!"
print("Alle Sanity Checks bestanden. ✓")


## 9. Abschließender Überblick

Hit Rate (In-DB) und Specificity (OOD) als Vorab-Überblick.
Die vollständige Auswertung erfolgt in NB 06.


In [ ]:
from metrics import compute_hit_rate, compute_specificity, compute_precision

hit_rate    = compute_hit_rate(results_df)
specificity = compute_specificity(results_df)
precision   = compute_precision(results_df)

print(f"Hit Rate (In-DB):  {hit_rate:.1%}")
print(f"Specificity (OOD): {specificity:.1%}")
print(f"Precision:         {precision:.1%}")
print()
print("Baseline (A_original, In-DB):")
base_hr = compute_hit_rate(results_df, filter_col="condition", filter_val="A_original")
print(f"  Hit Rate A_original: {base_hr:.1%}")
